## Step 2  
Input: s3://thesis--ec331-s3/enriched-volume-bids/  
Output: s3://thesis--ec331-s3/melted-volume-bids/  

TODO: I need to recognise what has been loaded and not to avoid duplication

In [ ]:
import pandas as pd
import awswrangler as wr
import time
from datetime import datetime
import gc
import os
import psutil
import boto3
from botocore.exceptions import ClientError

def get_memory_usage():
    """Return the current memory usage of the process in GB."""
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / (1024 ** 3)
    return memory_gb

# Columns we want to melt:
BAND_COLS = [f"BANDAVAIL{i}" for i in range(1, 11)]

def melt_and_write_chunks(df, chunk_size=50_000, output_folder="", file_label=""):
    """
    Melts the BANDAVAIL columns in chunks and writes each chunk directly to S3.
    - No checkpoint or manifest files
    - Filenames: {output_folder}/{file_label}_partXXXX.parquet

    Returns: (list_of_chunk_paths, total_rows_processed)
    """
    print(f"Starting streaming melt of DataFrame with shape: {df.shape}")
    print(f"Current memory usage: {get_memory_usage():.2f} GB")

    start_time = time.time()
    num_rows = len(df)
    num_chunks = (num_rows + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")

    total_rows_processed = 0
    chunk_paths = []

    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, num_rows)

        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk: {get_memory_usage():.2f} GB")

        # Copy out just this chunk
        chunk = df.iloc[chunk_start:chunk_end].copy()

        # Identify columns not in BAND_COLS
        id_vars_cols = [col for col in chunk.columns if col not in BAND_COLS]

        # Melt
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[c for c in BAND_COLS if c in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDVOLUME"
        )

        # Free memory from the chunk
        del chunk
        gc.collect()

        # Extract numeric band from e.g. "BANDAVAIL3"
        chunk_melted["BIDBAND"] = (
            chunk_melted["BIDBAND"]
            .str.extract(r"BANDAVAIL(\d+)")
            .astype(int)
        )

        # Optional type transformations
        if "BIDTYPE" in chunk_melted.columns:
            chunk_melted["BIDTYPE"] = chunk_melted["BIDTYPE"].astype(str)
        if "DUID" in chunk_melted.columns:
            chunk_melted["DUID"] = chunk_melted["DUID"].astype(str)
        if "SETTLEMENTDATE" in chunk_melted.columns:
            chunk_melted["SETTLEMENTDATE"] = pd.to_datetime(chunk_melted["SETTLEMENTDATE"])

        # Drop rows where BIDVOLUME is null
        chunk_melted.dropna(subset=["BIDVOLUME"], inplace=True)

        # Output path for this chunk
        chunk_len = len(chunk_melted)
        total_rows_processed += chunk_len

        # e.g.  s3://.../RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/enriched_volume_bids_20250312_102609_chunk1000of2123_part0001.parquet
        chunk_output = f"{output_folder.rstrip('/')}/{file_label}_part{i+1:04d}.parquet"
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            chunk_paths.append(chunk_output)
            print(f"  ✓ Wrote chunk {i+1}/{num_chunks} with {chunk_len} rows to {chunk_output}")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1}: {str(e)}")

        del chunk_melted
        gc.collect()
        print(f"Memory usage after chunk {i+1}: {get_memory_usage():.2f} GB")

    total_time = time.time() - start_time
    print(f"\nAll chunks processed in {total_time:.2f} seconds.")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB\n")

    return chunk_paths, total_rows_processed


def process_single_file(input_file, base_output_prefix):
    """
    Process a single Parquet file from S3:
    - Mirror its subfolder structure from 'enriched-volume-bids/'.
    - Instead of making an extra folder named after the file, the chunk outputs
      are placed directly in the same subfolder (no manifest/checkpoints).
    """
    print(f"\nProcessing single file: {input_file}")

    # Example input_file:
    #   s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/enriched_volume_bids_20250312_102609_chunk1000of2123.parquet

    bucket_name = input_file.split("/")[2]
    key = "/".join(input_file.split("/")[3:])  # e.g. "enriched-volume-bids/.../xxx.parquet"

    file_name = os.path.basename(key)               # e.g. "enriched_volume_bids_20250312_102609_chunk1000of2123.parquet"
    file_name_no_ext = os.path.splitext(file_name)[0] # e.g. "enriched_volume_bids_20250312_102609_chunk1000of2123"

    parent_path = os.path.dirname(key)  # e.g. "enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched"

    # We only want the portion after 'enriched-volume-bids/'
    # e.g. "RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched"
    prefix_remove = "enriched-volume-bids/"
    relative_subpath = parent_path[len(prefix_remove):].lstrip("/")

    # So the final output folder is:
    #   s3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/
    # We do NOT add the file_name_no_ext as a subfolder.
    if relative_subpath:
        output_folder = f"{base_output_prefix.rstrip('/')}/{relative_subpath}/"
    else:
        output_folder = f"{base_output_prefix.rstrip('/')}/"

    print(f"Output subfolder: {output_folder}")
    print(f"File label: {file_name_no_ext}")

    # Read entire Parquet
    start_time = time.time()
    df = wr.s3.read_parquet(path=input_file)
    print(f"Read file in {time.time() - start_time:.2f} seconds. Shape: {df.shape}")
    print(f"Memory usage after reading: {get_memory_usage():.2f} GB")

    # Melt in chunks
    chunk_files, total_rows = melt_and_write_chunks(
        df=df,
        chunk_size=50_000,
        output_folder=output_folder,
        file_label=file_name_no_ext
    )

    del df
    gc.collect()

    print(f"File done. Created {len(chunk_files)} chunk(s), total {total_rows} rows processed.")
    return chunk_files, total_rows


if __name__ == "__main__":
    print("PROCESSING VOLUME BIDS (No checkpoints, no manifests)")
    print("INPUT: s3://thesis--ec331-s3/enriched-volume-bids/<subfolders>/<parquet-files>")
    print("OUTPUT: s3://thesis--ec331-s3/melted-volume-bids/<matching-subfolders>/<chunks>\n")

    input_bucket = "thesis--ec331-s3"
    input_prefix = "enriched-volume-bids/"
    output_prefix = "s3://thesis--ec331-s3/melted-volume-bids/"

    s3_client = boto3.client("s3")
    paginator = s3_client.get_paginator("list_objects_v2")
    page_iterator = paginator.paginate(Bucket=input_bucket, Prefix=input_prefix)

    parquet_files = []
    for page in page_iterator:
        if "Contents" in page:
            for obj in page["Contents"]:
                key = obj["Key"]
                if key.endswith(".parquet"):
                    parquet_files.append(f"s3://{input_bucket}/{key}")

    print(f"\nFound {len(parquet_files)} .parquet file(s) under {input_prefix}.\n")

    total_files_processed = 0
    overall_rows = 0

    for file_s3_path in parquet_files:
        print("=" * 80)
        print(f"Starting to process: {file_s3_path}")
        try:
            melted_files, row_count = process_single_file(file_s3_path, base_output_prefix=output_prefix)
            print(f"Finished: {row_count} rows, {len(melted_files)} chunk file(s).\n")
            total_files_processed += 1
            overall_rows += row_count
        except Exception as ex:
            print(f"Error: {ex}")
        print("=" * 80)

    print(f"\nAll done. Processed {total_files_processed} file(s) total. "
          f"Overall rows processed: {overall_rows}\n")

PROCESSING VOLUME BIDS (No checkpoints, no manifests)
INPUT: s3://thesis--ec331-s3/enriched-volume-bids/<subfolders>/<parquet-files>
OUTPUT: s3://thesis--ec331-s3/melted-volume-bids/<matching-subfolders>/<chunks>


Found 2123 .parquet file(s) under enriched-volume-bids/.

Starting to process: s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/enriched_volume_bids_20250312_102609_chunk1000of2123.parquet

Processing single file: s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/enriched_volume_bids_20250312_102609_chunk1000of2123.parquet
Output subfolder: s3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/
File label: enriched_volume_bids_20250312_102609_chunk1000of2123
Read file in 0.22 seconds. Shape: (10000, 47)
Memory usage after reading: 0.19 GB
Starting streaming melt of DataFrame with shape: (10000, 47)
Current memory usage: 0.19 GB
Process

In [4]:
import awswrangler as wr
files = wr.s3.list_objects("s3://thesis--ec331-s3/enriched-volume-bids/")
print(f"Found {len(files)} files")

Found 2123 files
